In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import timm
import glob
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

class Config:
    SR = 32000
    DURATION = 5
    MAX_LENGTH = SR * DURATION
    N_MELS = 128
    N_FFT = 1024
    HOP_LENGTH = 512
    TARGET_FRAMES = 320
    BATCH_SIZE = 1
    MODEL_NAME = 'vit_small_patch16_224'
    WEIGHTS_PATH = glob.glob("/kaggle/input/notebooks/sofiasampara/notebook531018acac/best_transformer_fold0.pth")[0] 

BASE = "/kaggle/input/competitions/birdclef-2026"
TEST_DIR = os.path.join(BASE, "test_soundscapes")
SAMPLE_SUB = pd.read_csv(os.path.join(BASE, "sample_submission.csv"))
TAXONOMY = pd.read_csv(os.path.join(BASE, "taxonomy.csv"))

CLASSES = TAXONOMY['primary_label'].unique().tolist()
NUM_CLASSES = len(CLASSES)
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

In [3]:
class BirdCLEFTransformerSED(nn.Module):
    def __init__(self, model_name=Config.MODEL_NAME, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0, dynamic_img_size=True, global_pool='')
        in_features = self.backbone.num_features
        self.fc1 = nn.Linear(in_features, in_features)
        self.fc_prob = nn.Linear(in_features, num_classes)
        self.fc_att = nn.Linear(in_features, num_classes)

    def forward(self, x):
        tokens = self.backbone(x)
        if tokens.shape[1] == (Config.N_MELS // 16) * (Config.TARGET_FRAMES // 16) + 1:
            tokens = tokens[:, 1:, :] 
        x = torch.relu(self.fc1(tokens))
        framewise_probs = torch.sigmoid(self.fc_prob(x))
        framewise_att = torch.softmax(self.fc_att(x), dim=1)
        clipwise_probs = torch.sum(framewise_probs * framewise_att, dim=1) 
        return clipwise_probs

class TestDataset(Dataset):
    def __init__(self, audio_path, config):
        self.audio_path = audio_path
        self.config = config
        self.waveform, _ = torchaudio.load(audio_path)
        if self.waveform.shape[0] > 1:
            self.waveform = torch.mean(self.waveform, dim=0, keepdim=True)
            
        self.total_seconds = self.waveform.shape[1] // config.SR
        self.chunks = self.total_seconds // config.DURATION
        
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=config.SR, n_fft=config.N_FFT, hop_length=config.HOP_LENGTH,
            n_mels=config.N_MELS, f_min=50, f_max=14000
        )
        self.amp_to_db = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return self.chunks

    def __getitem__(self, idx):
        start = idx * self.config.MAX_LENGTH
        end = start + self.config.MAX_LENGTH
        chunk = self.waveform[:, start:end]
        
        mel_spec = self.amp_to_db(self.mel_transform(chunk))
        mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
        mel_spec = mel_spec * 2 - 1

        if mel_spec.shape[2] < self.config.TARGET_FRAMES:
            mel_spec = F.pad(mel_spec, (0, self.config.TARGET_FRAMES - mel_spec.shape[2]))
        else:
            mel_spec = mel_spec[:, :, :self.config.TARGET_FRAMES]
            
        return mel_spec.expand(3, -1, -1), (idx + 1) * 5

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BirdCLEFTransformerSED().to(device)
model.load_state_dict(torch.load(Config.WEIGHTS_PATH, map_location=device))
model.eval()

test_audio_files = glob.glob(f"{TEST_DIR}/*.ogg")
if len(test_audio_files) == 0:
    test_audio_files = []

all_results = []

with torch.no_grad():
    for audio_path in tqdm(test_audio_files):
        audio_id = os.path.basename(audio_path).replace(".ogg", "")
        dataset = TestDataset(audio_path, Config)
        loader = DataLoader(dataset, batch_size=Config.BATCH_SIZE, shuffle=False)
        
        for specs, end_times in loader:
            specs = specs.to(device)
            probs = model(specs).cpu().numpy()
            
            for i in range(len(probs)):
                row_id = f"{audio_id}_{end_times[i].item()}"
                res = {'row_id': row_id}
                for j, bird in enumerate(CLASSES):
                    res[bird] = probs[i][j]
                all_results.append(res)

if all_results:
    submission_df = pd.DataFrame(all_results)
else:
    submission_df = SAMPLE_SUB

submission_df.to_csv("submission.csv", index=False)
print("Submission saved!")
submission_df.head()

0it [00:00, ?it/s]

Submission saved!


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
